In [1]:
!pip install arxiv

     ---------------------------------------- 0.0/81.3 kB ? eta -:--:--
     ----- ---------------------------------- 10.2/81.3 kB ? eta -:--:--
     -------------------------------------- 81.3/81.3 kB 756.4 kB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Running setup.py install for sgmllib3k: started
  Running setup.py install for sgmllib3k: finished with status 'done'


  DEPRECATION: sgmllib3k is being installed using the legacy 'setup.py install' method, because it does not have a 'pyproject.toml' and the 'wheel' package is not installed. pip 23.1 will enforce this behaviour change. A possible replacement is to enable the '--use-pep517' option. Discussion can be found at https://github.com/pypa/pip/issues/8559

[notice] A new release of pip is available: 23.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

wikipedia_api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars=200)
wiki_tool= WikipediaQueryRun(
    api_wrapper=wikipedia_api_wrapper
    # name="WikipediaQueryRun",
    # description="Useful for answering questions about Wikipedia articles. Input should be a search query.",
)

In [13]:
wiki_tool.name

'wikipedia'

In [17]:
# WEb base loader 
from langchain_community.document_loaders import WebBaseLoader 
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

# 1. Load web documents
loader = WebBaseLoader("https://docs.smith.langchain.com/")
web_docs = loader.load()

# 2. Split documents into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents = splitter.split_documents(web_docs)

# 3. Create FAISS vector store using Google Embedding model
embedding_model = GoogleGenerativeAIEmbeddings(model="models/embedding-001")
vector_db = FAISS.from_documents(documents, embedding_model)

# 4. Create a retriever
retriever = vector_db.as_retriever()
retriever

VectorStoreRetriever(tags=['FAISS', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000029C9F032B00>, search_kwargs={})

In [24]:
retriever.name

In [22]:
from langchain.tools.retriever import create_retriever_tool

retriever_tool = create_retriever_tool(
    retriever,
    "Langsmith_search",
    "Search for infromation about Langsmith. For any Question, you can use this tool to search for relevant information in Langsmith documentation. Input should be a search query."
)

In [23]:
retriever_tool.name

'Langsmith_search'

In [25]:
# Arxiv tool 
from langchain_community.utilities import ArxivAPIWrapper
from langchain_community.tools import ArxivQueryRun

arxiv_wrapper = ArxivAPIWrapper(top_k_results=1, doc_content_chars_max=200)
arxiv = ArxivQueryRun(api_wrapper=arxiv_wrapper)
arxiv.name

'arxiv'

In [26]:
tools = [wiki_tool, arxiv, retriever_tool]

In [27]:
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'e:\\Langchain\\langchang + fastapi\\myvenv\\lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=4000)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=200)),
 Tool(name='Langsmith_search', description='Search for infromation about Langsmith. For any Question, you can use this tool to search for relevant information in Langsmith documentation. Input should be a search query.', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=functools.partial(<function _get_relevant_documents at 0x0000029C9F006560>, retriever=Vect

In [78]:
import os
from dotenv import load_dotenv 
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

llm = ChatGoogleGenerativeAI(
    model ="gimini-1.5-flash",
    temperature=0.1,
)


In [79]:
from langchain import hub

prompt = hub.pull("hwchase17/react")  # 🔥 Works perfectly
prompt


e:\Langchain\langchang + fastapi\myvenv\lib\site-packages\langsmith\client.py:272: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [80]:
# """Agent : # 1. Define the tools
# 2. Define the agent
# 3. Run the agent with a query
# 4. Print the response
# # """
from langchain_google_genai import ChatGoogleGenerativeAI  # ✅ Supports tool calling 
from langchain.agents import create_tool_calling_agent

agent = create_tool_calling_agent(llm=llm, tools=tools, prompt=prompt)

In [81]:
# Agent Executer 
from langchain.agents import AgentExecutor 

agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)
agent_executor

AgentExecutor(verbose=True, agent=RunnableMultiActionAgent(runnable=RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: message_formatter(x['intermediate_steps']))
})
| PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original inpu

In [82]:
tool_names = [tool.name for tool in tools]
response = agent_executor.invoke({
    "input": "What is Langsmith?",
    "tool_names": ", ".join(tool_names),
    "tools": "\n".join([f"{tool.name}: {tool.description}" for tool in tools]),
    "agent_scratchpad": ""
})
print(response)



> Entering new AgentExecutor chain...


Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised NotFound: 404 models/gimini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods..


NotFound: 404 models/gimini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.

In [11]:
# # test it 
# query = "What is langsmith?"
# docs = retriever.invoke(query)

# # Print preview of the top match
# print(docs[0].page_content[:300])


In [12]:
# for i, doc in enumerate(docs[:3]):
#     print(f"\n--- Chunk {i+1} ---")
#     print(doc.page_content[:500])
